In [1]:
import argparse
import sys
from pathlib import Path
from datetime import datetime

TEMPLATE_README = '''# {repo_name}

This repository contains ingestion pipelines for {client}.

## Structure
- common/: Shared modules (config, logger, utils)
- libs/: Zipped shared code (for Glue extra-py-files)
- scripts/: Pipeline scripts for Glue and other services
- scripts/athena/: Athena query SQLs
- workflows/: Glue Workflow definitions
- infrastructure/: IaC placeholders
- tests/: Test stubs/notebooks

## Naming Convention
All resource and folder names are generated from runtime args:
- client, env, domain, entity, source, connect, target, action

## How to scaffold a new pipeline job
Run:  
python scaffold.py --client acme --env dev --domain sales --entity order --source sap --target s3 --action ingest --connect appflow [--repo-prefix code/python-spark-sql] [--repo-root setup-guide]
'''

SCRIPT_TEMPLATE = '''"""
{script_filename}

Glue ingestion script placeholder for:
client={client}, domain={domain}, entity={entity}, source={source}, connect={connect}, target={target}, action={action}
"""
from common.config import parse_args
from common.logger import get_logger
from common.utils import build_s3_ingest_path

def run():
    args = parse_args()
    logger = get_logger()
    logger.info("Starting pipeline", extra=vars(args))
    # TODO: implement pipeline logic

if __name__ == "__main__":
    run()
'''

WORKFLOW_TEMPLATE = '''# {domain}_workflow.py

# Glue workflow definition placeholder for {domain}
'''

ATHENA_SQL_PLACEHOLDER = """-- Athena SQL queries for {domain}-{entity}-{env}
-- Place transformation/aggregation queries here
"""

NB_TEST_STUB = '''{
    "cells": [
    {
    "cell_type": "code",
    "execution_count": null,
    "id": "b7b4f1d0",
    "metadata": {},
    "outputs": [],
    "source": []
    }
    ],
    "metadata": {
    "language_info": {
    "name": "python"
    }
    },
    "nbformat": 4,
    "nbformat_minor": 5
    }'''

def create_repo_structure(args, force=False):
    client, env, domain, entity, source, target, action, connect, repo_prefix, repo_root = (
        args.client, args.env, args.domain, args.entity, args.source,
        args.target, args.action, args.connect, args.repo_prefix, args.repo_root
    )
    # Compose full repo root as [repo_root/][repo_prefix/]<client>-datalake-pipelines
    path_parts = []
    if repo_root:
        path_parts.append(repo_root)
    if repo_prefix:
        path_parts.append(repo_prefix)
    path_parts.append(f'{client}-datalake-pipelines')
    root = Path(*path_parts)
    if root.exists():
        if force:
            import shutil
            shutil.rmtree(root)
        else:
            sys.exit(f"Error: Repository root '{root}' already exists. Use --force to overwrite.")
    root.mkdir(parents=True)

    # Root README
    (root / 'README.md').write_text(
        TEMPLATE_README.format(repo_name=root.name, client=client)
    )

    # common/
    common = root / 'common'
    common.mkdir()
    for fname, stub in [
        ('__init__.py', '"""Common module init"""\n'),
        ('config.py', '"""Config helpers for pipeline jobs"""\n'),
        ('logger.py', '"""Central logging setup (use Python logging)"""\n'),
        ('utils.py', '"""Utility functions (S3 path builders, name generators)"""\n')
    ]:
        (common / fname).write_text(stub)

    # libs/
    libs = root / 'libs'
    libs.mkdir()
    zipname = f"{client}-common.zip"
    (libs / zipname).touch()

    # scripts/glue/<job-folder>
    scripts = root / 'scripts' / 'glue'
    scripts.mkdir(parents=True)
    folder_name = f"{client}-{domain}-{entity}-{source}-{connect}-{target}-{action}"
    pipeline_dir = scripts / folder_name
    pipeline_dir.mkdir()

    script_filename = f"{folder_name}.py".replace('-', '_')
    script_file = pipeline_dir / script_filename
    script_file.write_text(
        SCRIPT_TEMPLATE.format(
            script_filename=script_filename,
            client=client,
            domain=domain,
            entity=entity,
            source=source,
            connect=connect,
            target=target,
            action=action
        )
    )
    (pipeline_dir / 'requirements.txt').write_text("# Add Python requirements if needed\n")

    # scripts/athena/<entity>.sql (Athena SQL placeholder)
    athena_dir = root / 'scripts' / 'athena'
    athena_dir.mkdir(exist_ok=True, parents=True)
    athena_sql_file = athena_dir / f"{domain}_{entity}_{env}.sql"
    athena_sql_file.write_text(ATHENA_SQL_PLACEHOLDER.format(domain=domain, entity=entity, env=env))

    # workflows/
    workflows = root / 'workflows'
    workflows.mkdir()
    wf_file = workflows / f"{domain}_workflow.py"
    wf_file.write_text(WORKFLOW_TEMPLATE.format(domain=domain))

    # infrastructure/
    infra = root / 'infrastructure'
    infra.mkdir()
    (infra / 'README.md').write_text("# Infrastructure-as-Code placeholders\n")

    # tests/
    tests = root / 'tests'
    tests.mkdir()
    test_file = tests / f"nbtest_{folder_name}.ipynb".replace('-', '_')
    test_file.write_text(NB_TEST_STUB)

    print(f"Scaffolded repository at {root.resolve()}")

def generate_s3_names(args):
    client, env, domain, entity, source, target, action, connect = (
        args.client, args.env, args.domain, args.entity, args.source,
        args.target, args.action, args.connect
    )
    zones = ["ingest", "structured", "scripts", "temp", "archive", "metadata"]
    buckets = {zone: f"{client}-datalake-{env}-{zone}" for zone in zones}
    ts = datetime.now().strftime("%Y%m%d%H%M%S")
    date_iso = datetime.now().strftime("%Y-%m-%d")
    filter_field = "filter"
    filter_value = "filter_value"
    prefixes = {
        "ingest": [
            f"s3://{buckets['ingest']}/{source}/{entity}/",
            f"s3://{buckets['ingest']}/{source}/{entity}/{filter_field}={filter_value}/{entity}_{ts}.csv"
        ],
        "structured": [
            f"s3://{buckets['structured']}/{source}/{entity}/",
            f"s3://{buckets['structured']}/{source}/{entity}/{filter_field}={filter_value}/{entity}_{ts}.csv"
        ],
        "scripts": [
            f"s3://{buckets['scripts']}/glue/{client}-{domain}-{entity}-{source}-{connect}-{target}-{action}/",
            f"s3://{buckets['scripts']}/glue/{domain}{entity}{source}{connect}{target}_{action}/{client}-{domain}-{entity}-{source}-{connect}-{target}-{action}.py"
        ],
        "temp": [
            f"s3://{buckets['temp']}/athena-query-results/",
            f"s3://{buckets['temp']}/intermediate/"
        ],
        "archive": [
            f"s3://{buckets['archive']}/{source}/{entity}/{ts}/"
        ],
        "metadata": [
            f"s3://{buckets['metadata']}/runs/{source}/{entity}/run_date={date_iso}/summary-RUNID.json",
            f"s3://{buckets['metadata']}/runs/{source}/{entity}/run_date={date_iso}/details-RUNID.json",
            f"s3://{buckets['metadata']}/configs/source_configs.json",
            f"s3://{buckets['metadata']}/schemas/{source}-{entity}-schema.json",
            f"s3://{buckets['metadata']}/lineage/{source}-{entity}-lineage.json"
        ]
    }
    print("\nS3 Bucket Names:")
    for zone, name in buckets.items():
        print(f"  {zone}: {name}")
    print("\nExample S3 Paths:")
    for zone, paths in prefixes.items():
        print(f"\n{zone.capitalize()} Zone:")
        for p in paths:
            print(f"  {p}")

def generate_glue_names(args):
    client, env, domain, entity, source, target, action, connect = (
        args.client, args.env, args.domain, args.entity, args.source,
        args.target, args.action, args.connect
    )
    glue_job = f"gluejob_{env}_{client}_{domain}_{entity}_{source}_{connect}_{target}_{action}"
    glue_workflow = f"gluewf_{env}_{client}_{domain}_{entity}_{source}_{connect}_{target}_{action}"
    glue_crawler = f"gluecr_{env}_{client}_{domain}_{entity}_{source}_{connect}_{target}_{action}"
    glue_job_folder = f"{client}-{domain}-{entity}-{source}-{connect}-{target}-{action}"
    script_name = f"{client}{domain}{entity}{source}{connect}{target}{action}.py"
    catalog_db_ingest = f"{client}_{env}_ingest"
    catalog_db_structured = f"{client}_{env}_structured"
    catalog_db_metadata = f"{client}_{env}_metadata"
    catalog_table = f"{domain}_{entity}"
    print("glue_job:", glue_job)
    print("glue_workflow:", glue_workflow)
    print("glue_crawler:", glue_crawler)
    print("glue_job_folder:", glue_job_folder)
    print("script_name:", script_name)
    print("catalog_db_ingest:", catalog_db_ingest)
    print("catalog_db_structured:", catalog_db_structured)
    print("catalog_db_metadata:", catalog_db_metadata)
    print("catalog_table:", catalog_table)

def initialize_names(cmd_args):
    parser = argparse.ArgumentParser(description="Generate AWS Glue service names based on conventions.")
    parser.add_argument("--client", type=str, default="myclient", help="Client name")
    parser.add_argument("--env", type=str, default="dev", help="Environment")
    parser.add_argument("--domain", type=str, default="schema", help="Domain area")
    parser.add_argument("--entity", type=str, default="table", help="Entity/table name")
    parser.add_argument("--source", type=str, default="sapappflow", help="Data source")
    parser.add_argument("--target", type=str, default="s3", help="Target destination")
    parser.add_argument("--action", type=str, default="ingest", help="Action type")
    parser.add_argument("--connect", type=str, default="appflow", help="Connection type")
    parser.add_argument("--repo-prefix", type=str, default=None, help="Optional: subdirectory prefix, e.g. code/python-spark-sql")
    parser.add_argument("--repo-root", type=str, default=None, help="Optional: repo root, e.g. setup-guide")
    args = parser.parse_args(cmd_args)
    return args

if __name__ == "__main__":
    runtime_args = [
        "--client", "client",
        "--env",    "env",
        "--domain", "domain",
        "--entity", "entity",
        "--source", "source",
        "--target", "target",
        "--action", "action",
        "--connect", "connect",
        # "--repo-prefix", "code/python-spark-sql",
        "--repo-root", "setup-guide"
    ]
    args = initialize_names(runtime_args)
    generate_glue_names(args)
    generate_s3_names(args)
    create_repo_structure(args)


glue_job: gluejob_env_client_domain_entity_source_connect_target_action
glue_workflow: gluewf_env_client_domain_entity_source_connect_target_action
glue_crawler: gluecr_env_client_domain_entity_source_connect_target_action
glue_job_folder: client-domain-entity-source-connect-target-action
script_name: clientdomainentitysourceconnecttargetaction.py
catalog_db_ingest: client_env_ingest
catalog_db_structured: client_env_structured
catalog_db_metadata: client_env_metadata
catalog_table: domain_entity

S3 Bucket Names:
  ingest: client-datalake-env-ingest
  structured: client-datalake-env-structured
  scripts: client-datalake-env-scripts
  temp: client-datalake-env-temp
  archive: client-datalake-env-archive
  metadata: client-datalake-env-metadata

Example S3 Paths:

Ingest Zone:
  s3://client-datalake-env-ingest/source/entity/
  s3://client-datalake-env-ingest/source/entity/filter=filter_value/entity_20250518165057.csv

Structured Zone:
  s3://client-datalake-env-structured/source/entity/